# ToadWatch_PR - START HERE

## What changed and why
In the original study (**v1**) the combined model scored slightly *below* the call-only model (F1 0.989 vs 0.991 on 818 test records - a 2-case gap). v1 is frozen and its test set is used up, so I did **not** tune anything against it. The follow-up (**v2**) fixes the causes *fairly*:

| Problem found in v1 | v2 fix | Why it is not cheating |
|---|---|---|
| Call model leaned on length / punctuation counts (dataset style, not scam content) | those columns are dropped from **every** model | makes the *strong* model weaker/honest, not the combined one stronger |
| Combined model only concatenated two feature sets (weak channel dilutes the strong one) | a **stacked late-fusion** variant: one forest per channel + a small logistic meta-learner trained only on out-of-fold probabilities. Cross-channel features (shared brand, both urgent...) were also tested; they gave no CV gain, so they are **off** (`USE_CROSS_FEATURES`) | the meta-learner never sees a row's own label-fit prediction; nothing uses the test set |
| One shared setting for all models | every model tuned with the **same grid and CV procedure** | standard fair comparison |
| Single split, 2-case differences read as "better" | repeated CV, bootstrap intervals, paired tests (McNemar) | measures noise instead of ignoring it |
| Early vs stacked is a design choice | picked by CV on train+val with a rule written **before** the test (`TIE_MARGIN`) | test never influences the choice |
| Seen v1 test results | new seed -> new split, new draw of messages, new pairing; test locked and fingerprinted; runs **once** | v2 test is a separate confirmatory run |

I cannot promise the combined model will win - if it does not, the notebooks will say so.

## Run order (VS Code)
1. Open the folder **`ToadWatch_PR`** in VS Code (File -> Open Folder).
2. Top right of a notebook: **Select Kernel -> Python Environments** -> pick the Python that has pandas / scikit-learn (the anaconda one). Install once if needed: `pip install -r requirements.txt`.
3. Run these notebooks **in order** (each one: *Run All*):

| # | Notebook | What it does | Time |
|---|---|---|---|
| 0 | `00_START_HERE.ipynb` | environment check (this file) | 5 s |
| – | `v2_00_explore_dataset.ipynb` | *optional, any time after step 1:* browse the raw files and the train / validation / test sets (tables, charts, search) | 20 s |
| 1 | `v2_01_prepare_data.ipynb` | load, clean, dedupe, split 70/10/20, pair, lock test set | ~2 min |
| 2 | `v2_02_diagnose_combined.ipynb` | *why* does combined lag? ablations, complementarity (train+val CV) | ~8 min |
| 3 | `v2_03_tune_and_choose.ipynb` | tune every model, pick early vs stacked, freeze config, learning curve | ~15 min |
| 4 | `v2_04_final_test_once.ipynb` | **one-shot** test, CIs, paired tests, test charts | ~1 min |
| 5 | `v2_05_charts_gallery.ipynb` | draws every chart (also saved to `outputs/v2/figures`) | ~3 min |

The older notebooks `01`-`06` are the frozen **v1** study; you do not need to rerun them.

**Automatic documents:** the last cell of `v2_04` and `v2_05` regenerates the Word report + PDF and the PowerPoint + PDF into `report/v2/` from the run you just made (all numbers, tables, charts and the verdict are read from the saved results).

**Prefer the terminal?** `python scripts/run_v2.py all` runs steps 1-3 and the charts; `python scripts/run_v2.py final` is the one-shot test.

**Look at the data:** `ToadWatch_PR/dataset/v2/{train,validation,test}/` holds plain CSV files (`paired_records.csv`, `calls.csv`, `messages.csv`) for each set; `v2_00_explore_dataset.ipynb` displays them.

Results land in `outputs/v2/tables`, charts in `outputs/v2/figures`, test set and splits in `data/processed/v2`.

In [1]:
import sys, importlib
from pathlib import Path
print("python", sys.version.split()[0])
for m in ["pandas","numpy","sklearn","scipy","matplotlib","seaborn","pyarrow","joblib"]:
    try: print(f"  ok   {m} {importlib.import_module(m).__version__}")
    except Exception as e: print(f"  MISSING {m}  ->  pip install {m}")
root = Path.cwd().parent
print("project folder:", root, "| notebooks folder name ok:", Path.cwd().name == "notebooks")
for d in ["Call Safe data","Call Dangerous data","SMS, Email, URL Safe","SMS, Email, URL Dangerous"]:
    print(("  ok   " if (root.parent / d).exists() else "  MISSING ") + d)

python 3.14.6


  ok   pandas 3.0.3
  ok   numpy 2.4.6


  ok   sklearn 1.9.0
  ok   scipy 1.18.0
  ok   matplotlib 3.11.0


  ok   seaborn 0.13.2
  ok   pyarrow 23.0.1
  ok   joblib 1.5.3
project folder: /home/rush/Downloads/Fase/ToadWatch_PR | notebooks folder name ok: True
  ok   Call Safe data
  ok   Call Dangerous data
  ok   SMS, Email, URL Safe
  ok   SMS, Email, URL Dangerous
